# Cross-Validation — Model Evaluation

## What is Cross-Validation?

Cross-Validation (CV) is a technique used to evaluate how well a machine-learning model is likely to perform on **unseen data**.

Instead of evaluating the model using only one train/validation split, we train and validate it multiple times using different portions of the training data.

---

# Why Do We Need Cross-Validation?

A single split might give:

```text id="0lbh1n"
Train → 80%
Test  → 20%
```

But the result depends partly on which rows happened to enter each group.

For example:

```text id="xkn13o"
Split 1 → R² = 0.81
Split 2 → R² = 0.74
Split 3 → R² = 0.86
```

So judging the model from only one split may give us an incomplete picture.

Cross-validation gives us a more reliable estimate by evaluating the model across multiple splits.

---

# K-Fold Cross-Validation

One of the most common approaches is **K-Fold Cross-Validation**.

Suppose:

```text id="cnz6qw"
K = 5
```

The training data is divided into 5 groups called **folds**.

```text id="5r94so"
DATA

┌────┬────┬────┬────┬────┐
│ F1 │ F2 │ F3 │ F4 │ F5 │
└────┴────┴────┴────┴────┘
```

The model is trained **5 times**.

### Round 1

```text id="f7n4n0"
F1 = Validation

F1      F2      F3      F4      F5
TEST    TRAIN   TRAIN   TRAIN   TRAIN
```

### Round 2

```text id="v19c09"
F1      F2      F3      F4      F5
TRAIN   TEST    TRAIN   TRAIN   TRAIN
```

### Round 3

```text id="9mtgg0"
F1      F2      F3      F4      F5
TRAIN   TRAIN   TEST    TRAIN   TRAIN
```

And so on until every fold has been used once for validation.

> Inside cross-validation, this held-out fold is more precisely called the **validation fold**, not the final test set.

---

# Final CV Score

Suppose 5-fold CV produces:

```text id="dgrfhg"
Fold 1 → R² = 0.80
Fold 2 → R² = 0.76
Fold 3 → R² = 0.82
Fold 4 → R² = 0.79
Fold 5 → R² = 0.81
```

Then we usually inspect:

```text id="i11cln"
Mean CV R² ≈ 0.796
```

We should also look at the variation between folds.

For example:

```text id="vnpfuo"
Mean = 0.80
Std  = 0.02
```

Small variation suggests relatively stable performance across these folds.

---

# Train / Validation / Test Idea

A clean ML workflow is:

```text id="t2gbtl"
FULL DATASET
     │
     ▼
Train/Test Split
     │
     ├─────────────── TEST SET
     │                Keep untouched
     │
     ▼
TRAINING DATA
     │
     ▼
Cross-Validation
     │
     ├── Fold 1
     ├── Fold 2
     ├── Fold 3
     ├── Fold 4
     └── Fold 5
     │
     ▼
Model / Hyperparameter Selection
     │
     ▼
Train final model
     │
     ▼
TEST SET
     │
     ▼
Final unbiased evaluation
```

### Important Rule

> **Use cross-validation on the training data for model/hyperparameter decisions. Keep the final test set untouched until the end.**

---

# Why Pipeline Matters With Cross-Validation

Suppose we need:

```text id="udax3l"
StandardScaler
      ↓
Model
```

We should put preprocessing inside a Pipeline:

```text id="kh8kno"
Pipeline
   │
   ├── StandardScaler
   │
   └── Model
```

During every CV fold:

```text id="hw82hj"
Training portion
      ↓
Scaler.fit()
      ↓
Model.fit()


Validation portion
      ↓
Scaler.transform()
      ↓
Model.predict()
```

This prevents preprocessing information from leaking from the validation fold into training.

---

# Data Leakage

### Wrong Approach ❌

```text id="5jfs8m"
FULL DATA
    ↓
StandardScaler.fit_transform()
    ↓
Cross-Validation
```

The scaler has already learned information from all rows.

That means validation information can leak into preprocessing.

---

### Correct Approach ✅

```text id="c4qyr5"
RAW TRAINING DATA
       ↓
Cross-Validation
       ↓
Pipeline fitted separately
inside each training fold
       ↓
Scaler
       ↓
Model
```

---

# Basic Scikit-Learn Pattern

```python id="y3j8xg"
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="r2"
)
```

This returns one score for each fold.

Example:

```text id="i27yya"
[0.80, 0.76, 0.82, 0.79, 0.81]
```

Then:

```python id="w9v4as"
scores.mean()
```

gives the average CV score.

And:

```python id="hdnwxm"
scores.std()
```

shows how much the scores vary across folds.

---

# 🧠 Quick Recall Sketch

```text id="dw93qg"
                DATASET
                   │
                   ▼
            Train/Test Split
              /          \
             /            \
            ▼              ▼
       TRAIN DATA        TEST DATA
            │            untouched
            ▼
       CROSS-VALIDATION
            │
   ┌────────┼────────┐
   ▼        ▼        ▼
 Fold 1   Fold 2   Fold 3 ...
   │        │        │
   └────────┼────────┘
            ▼
      Average CV Score
            │
            ▼
    Choose model/settings
            │
            ▼
      Train final model
            │
            ▼
         TEST DATA
            │
            ▼
      Final Performance
```

---

# Cross-Validation vs Test Set

```text id="u58j43"
Cross-Validation
→ Used during development
→ Compare models
→ Tune hyperparameters


Test Set
→ Used at the end
→ Final evaluation
→ Should not guide tuning
```

---

# Key Interview Answer

> **Cross-validation evaluates a model across multiple train/validation splits instead of relying on one split. It provides a more reliable estimate of generalization and is commonly used for model selection and hyperparameter tuning while keeping the final test set untouched.**

---

# 10-Second Memory

```text id="ajdgb7"
ONE SPLIT
   ↓
Result may depend on split

K-FOLD
   ↓
Train K times
   ↓
Each fold validates once
   ↓
Mean + variation
   ↓
More reliable evaluation
```

## Golden Rule

> **CV helps us choose. The untouched test set gives the final check.**


In [1]:
from ucimlrepo import fetch_ucirepo

wine = fetch_ucirepo(id=186)

X = wine.data.features
y = wine.data.targets.squeeze()

In [2]:
X.head()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4


In [3]:
y.head()

0    5
1    5
2    5
3    6
4    5
Name: quality, dtype: int64

In [4]:
X.shape

(6497, 11)

In [5]:
y.shape

(6497,)

In [8]:
X.isnull().any()

fixed_acidity           False
volatile_acidity        False
citric_acid             False
residual_sugar          False
chlorides               False
free_sulfur_dioxide     False
total_sulfur_dioxide    False
density                 False
pH                      False
sulphates               False
alcohol                 False
dtype: bool

In [9]:
y.isnull().sum()

0

In [10]:
X.describe()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol
count,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000
mean,7.215307,0.339666,0.318633,5.443235,0.056034,30.525319,115.744574,0.994697,3.218501,0.531268,10.491801
std,1.296434,0.164636,0.145318,4.757804,0.035034,17.749400,56.521855,0.002999,0.160787,0.148806,1.192712
min,3.800000,0.080000,0.000000,0.600000,0.009000,1.000000,6.000000,0.987110,2.720000,0.220000,8.000000
25%,6.400000,0.230000,0.250000,1.800000,0.038000,17.000000,77.000000,0.992340,3.110000,0.430000,9.500000
50%,7.000000,0.290000,0.310000,3.000000,0.047000,29.000000,118.000000,0.994890,3.210000,0.510000,10.300000
75%,7.700000,0.400000,0.390000,8.100000,0.065000,41.000000,156.000000,0.996990,3.320000,0.600000,11.300000
max,15.900000,1.580000,1.660000,65.800000,0.611000,289.000000,440.000000,1.038980,4.010000,2.000000,14.900000


In [12]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed_acidity         6497 non-null   float64
 1   volatile_acidity      6497 non-null   float64
 2   citric_acid           6497 non-null   float64
 3   residual_sugar        6497 non-null   float64
 4   chlorides             6497 non-null   float64
 5   free_sulfur_dioxide   6497 non-null   float64
 6   total_sulfur_dioxide  6497 non-null   float64
 7   density               6497 non-null   float64
 8   pH                    6497 non-null   float64
 9   sulphates             6497 non-null   float64
 10  alcohol               6497 non-null   float64
dtypes: float64(11)
memory usage: 558.5 KB


In [13]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
print("X:", X.shape)
print("y:", y.shape)

X: (6497, 11)
y: (6497,)


In [15]:
print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (5197, 11)
Test : (1300, 11)


In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

model = Pipeline([
    ("scalar",StandardScaler()),
    ("model",LinearRegression())
])

In [17]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="r2"
)

In [18]:
print("Fold Scores:", scores)
print("Mean R²    :", scores.mean())
print("Std R²     :", scores.std())

Fold Scores: [0.28621092 0.31153757 0.29009954 0.28586232 0.30251639]
Mean R²    : 0.2952453498225641
Std R²     : 0.010143430747222178


In [21]:
from sklearn.model_selection import KFold

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [22]:
scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=kf,
    scoring="r2"
)

print("Scores :", scores)
print("Mean R²:", scores.mean())
print("Std R² :", scores.std())

Scores : [0.30159759 0.31899194 0.32031781 0.26583262 0.25330158]
Mean R²: 0.29200830770253755
Std R² : 0.02758578006828261


In [37]:
from sklearn.model_selection import cross_validate

cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=kf,
    scoring=[
        "r2",
        "neg_mean_absolute_error",
        "neg_root_mean_squared_error"
    ]
)

In [38]:
cv_results

{'fit_time': array([0.00999379, 0.0080018 , 0.00800347, 0.00998974, 0.00715351]),
 'score_time': array([0.00500894, 0.00400329, 0.00500369, 0.00499773, 0.00400424]),
 'test_r2': array([0.30159759, 0.31899194, 0.32031781, 0.26583262, 0.25330158]),
 'test_neg_mean_absolute_error': array([-0.58381777, -0.55897816, -0.56740446, -0.58064556, -0.56013282]),
 'test_neg_root_mean_squared_error': array([-0.76411587, -0.71166628, -0.72949062, -0.74408853, -0.73331584])}

In [39]:
cv_results.keys()

dict_keys(['fit_time', 'score_time', 'test_r2', 'test_neg_mean_absolute_error', 'test_neg_root_mean_squared_error'])

In [40]:
r2 = cv_results["test_r2"]

mae = -cv_results["test_neg_mean_absolute_error"]

rmse = -cv_results["test_neg_root_mean_squared_error"]

In [41]:
print("CV R²  :", r2.mean())
print("CV MAE :", mae.mean())
print("CV RMSE:", rmse.mean())

CV R²  : 0.29200830770253755
CV MAE : 0.5701957549112577
CV RMSE: 0.7365354275381627


# GridSearchCV — Hyperparameter Tuning

## What is GridSearchCV?

GridSearchCV automatically searches through different **hyperparameter combinations** and uses **Cross-Validation** to determine which combination performs best.

Instead of manually doing:

```text
Try alpha = 0.001
Train
Evaluate

Try alpha = 0.01
Train
Evaluate

Try alpha = 0.1
Train
Evaluate

...
```

GridSearchCV automates the process.

---

# What is a Hyperparameter?

A hyperparameter is a setting we choose **before/during model training** rather than a parameter learned directly from the training data.

Examples:

```text
Ridge
→ alpha

Lasso
→ alpha

Elastic Net
→ alpha
→ l1_ratio

PolynomialFeatures
→ degree

Decision Tree
→ max_depth

Random Forest
→ n_estimators
→ max_depth
```

The model learns things such as:

```text
coefficients
intercept
```

But we choose/tune things such as:

```text
alpha
l1_ratio
degree
max_depth
```

---

# Why GridSearchCV?

Suppose:

```text
alpha
→ [0.001, 0.01, 0.1, 1]

l1_ratio
→ [0.2, 0.5, 0.8]
```

GridSearchCV tests every combination:

```text
                 l1_ratio

              .2     .5     .8
            ┌──────┬──────┬──────┐
α = .001    │  ✓   │  ✓   │  ✓   │
            ├──────┼──────┼──────┤
α = .01     │  ✓   │  ✓   │  ✓   │
            ├──────┼──────┼──────┤
α = .1      │  ✓   │  ✓   │  ✓   │
            ├──────┼──────┼──────┤
α = 1       │  ✓   │  ✓   │  ✓   │
            └──────┴──────┴──────┘

4 × 3
=
12 combinations
```

If:

```text
CV = 5
```

each combination is evaluated across 5 folds.

Conceptually:

```text
12 combinations
×
5 folds
=
60 model fits
```

GridSearchCV may perform an additional refit of the winning configuration on all supplied training data when `refit=True`, which is the default.

---

# Pipeline + GridSearchCV

This is the important workflow:

```text
RAW TRAINING DATA
        ↓
     Pipeline
        │
        ├── Preprocessing
        │
        └── Model
        ↓
   GridSearchCV
        ↓
Cross-Validation
        ↓
Try parameter combinations
        ↓
Compare CV scores
        ↓
Best Parameters
        ↓
Best Pipeline
        ↓
FINAL TEST SET
```

---

# Pipeline Parameter Syntax

Suppose:

```python
Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet())
])
```

The model step is named:

```text
model
```

ElasticNet contains:

```text
alpha
l1_ratio
```

Therefore GridSearchCV uses:

```text
model__alpha
model__l1_ratio
```

Notice the **double underscore `__`**.

---

# Easy Memory

```text
Pipeline Step
     +
     __
     +
Parameter


model + __ + alpha

        ↓

model__alpha
```

Examples:

```text
model__alpha

model__l1_ratio

model__max_depth

model__n_estimators
```

---

# GridSearchCV Pattern

```python
params = {
    "model__alpha": [0.001, 0.01, 0.1, 1],
    "model__l1_ratio": [0.2, 0.5, 0.8]
}
```

Then:

```python
GridSearchCV(
    estimator=pipeline,
    param_grid=params,
    cv=5,
    scoring="r2"
)
```

---

# What GridSearchCV Gives Us

After fitting:

```text
best_params_
     ↓
Best hyperparameter combination


best_score_
     ↓
Best mean cross-validation score


best_estimator_
     ↓
Best fitted Pipeline/model
```

Example:

```python
grid.best_params_

grid.best_score_

grid.best_estimator_
```

---

# Train / CV / Test Workflow

```text
FULL DATASET
      ↓
Train/Test Split
      │
      ├─────────────→ TEST 🔒
      │
      ▼
TRAIN DATA
      ↓
GridSearchCV
      ↓
Cross-Validation
      ↓
Choose hyperparameters
      ↓
Best estimator
      ↓
TEST 🔓
      ↓
Final evaluation
```

---

# Why Not Tune With Test Data?

If we repeatedly choose parameters based on test performance:

```text
Test alpha=.01
      ↓
Test alpha=.1
      ↓
Test alpha=1
      ↓
Choose best
```

then the test set has influenced model development.

It is no longer a clean final evaluation.

Instead:

```text
TRAIN
  ↓
Cross-Validation
  ↓
Tune
  ↓
Choose model
  ↓
TEST ONCE
```

---

# 🧠 Quick Recall Sketch

```text
                 GRIDSEARCHCV
                      │
          ┌───────────┴───────────┐
          ▼                       ▼
     PARAMETERS              CROSS-VALIDATION
          │                       │
     α / ratio / etc.            K-Fold
          │                       │
          └───────────┬───────────┘
                      ▼
               TRY COMBINATIONS
                      ↓
                COMPARE SCORES
                      ↓
               BEST PARAMETERS
                      ↓
                BEST ESTIMATOR
                      ↓
                  TEST SET
```

---

# Manual Loop vs GridSearchCV

```text
MANUAL

for alpha...
    for ratio...
        fit
        score


GRIDSEARCHCV

parameters
    ↓
GridSearchCV
    ↓
DONE
```

GridSearchCV also gives a structured `cv_results_` table containing the results for every candidate.

---

# GridSearchCV vs Cross-Validation

```text
Cross-Validation
       ↓
"How well does this model/settings
 generalize across folds?"


GridSearchCV
       ↓
"Which tested hyperparameter
 combination performs best
 according to CV?"
```

GridSearchCV **uses Cross-Validation internally**.

---

# Important Limitation

GridSearchCV tries **every specified combination**.

For a small search:

```text
4 × 3 = 12
```

this is fine.

But imagine:

```text
10 × 10 × 10 × 10
=
10,000 combinations
```

With 5-fold CV:

```text
10,000 × 5
=
50,000 fits
```

This can become expensive.

Later we will learn:

```text
RandomizedSearchCV
        ↓
Doesn't exhaustively test every combination
        ↓
Useful for larger search spaces
```

And later:

```text
Optuna
↓
More advanced hyperparameter optimization
```

---

# Key Interview Answer

> **GridSearchCV performs an exhaustive search over specified hyperparameter combinations using cross-validation. It selects the combination with the best mean validation score and can refit the best estimator on the full training data.**

---

# 10-Second Recall

```text
PARAMETERS
    ↓
GRID
    ↓
CROSS-VALIDATE
    ↓
COMPARE
    ↓
BEST PARAMS
    ↓
BEST MODEL
    ↓
FINAL TEST
```

## Golden Rule

> **Train data is used to choose. Test data is used to verify the final choice.**


Pipeline
   │
   ├── StandardScaler()
   └── ElasticNet(max_iter=10000)

In [47]:
from sklearn.linear_model import ElasticNet

EN_model = Pipeline([
    ("scalar",StandardScaler()),
    ("model",ElasticNet(max_iter=10000))
])

In [51]:
params = {
    "model__alpha":[0.001,0.01,0.1,1],
    "model__l1_ratio":[0.2,0.5,0.8]
}

In [52]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    estimator=EN_model,
    param_grid=params,
    cv=kf,
    scoring="r2"
)

In [53]:
grid.fit(X_train, y_train)

GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scalar', StandardScaler()),
                                       ('model', ElasticNet(max_iter=10000))]),
             param_grid={'model__alpha': [0.001, 0.01, 0.1, 1],
                         'model__l1_ratio': [0.2, 0.5, 0.8]},
             scoring='r2')

In [54]:
model.get_params().keys()

dict_keys(['memory', 'steps', 'verbose', 'scalar', 'model', 'scalar__copy', 'scalar__with_mean', 'scalar__with_std', 'model__copy_X', 'model__fit_intercept', 'model__n_jobs', 'model__positive'])

In [55]:
print("Best Parameters:", grid.best_params_)
print("Best CV R²:", grid.best_score_)

Best Parameters: {'model__alpha': 0.001, 'model__l1_ratio': 0.2}
Best CV R²: 0.29203446293783897


In [56]:
grid.best_estimator_

Pipeline(steps=[('scalar', StandardScaler()),
                ('model',
                 ElasticNet(alpha=0.001, l1_ratio=0.2, max_iter=10000))])

In [57]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

In [58]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

print("FINAL TEST")
print("R²  :", round(r2_score(y_test, y_pred), 3))
print("MAE :", round(mean_absolute_error(y_test, y_pred), 3))
print("RMSE:", round(root_mean_squared_error(y_test, y_pred), 3))

FINAL TEST
R²  : 0.26
MAE : 0.566
RMSE: 0.739


# RandomizedSearchCV — Faster Hyperparameter Search

## What is RandomizedSearchCV?

`RandomizedSearchCV` is a hyperparameter-tuning technique that tests a **limited number of randomly selected parameter combinations** using Cross-Validation.

It is similar to `GridSearchCV`, but it does **not have to test every possible combination**.

---

# GridSearchCV vs RandomizedSearchCV

Suppose we have:

```text
alpha
→ 10 possible values

l1_ratio
→ 10 possible values
```

GridSearchCV:

```text
10 × 10
=
100 combinations

CV = 5

100 × 5
=
500 CV fits
```

RandomizedSearchCV can instead say:

```text
"Just try 20 combinations"
```

Then:

```text
20 combinations
×
5 folds
=
100 CV fits
```

---

# Main Difference

```text
GridSearchCV
     ↓
Tests EVERY specified combination
     ↓
More exhaustive
     ↓
Can become expensive


RandomizedSearchCV
     ↓
Tests a chosen number of combinations
     ↓
Faster for large search spaces
     ↓
May miss the absolute best combination
```

---

# The Important Parameter — n_iter

```python
n_iter=20
```

means:

> Try 20 parameter combinations.

Example:

```text
Possible combinations = 1000

n_iter = 20

        ↓

Only 20 combinations are sampled
```

With:

```text
CV = 5
```

we get:

```text
20 × 5
=
100 CV fits
```

---

# Pipeline + RandomizedSearchCV

Same Pipeline idea:

```text
RAW DATA
   ↓
Pipeline
   │
   ├── StandardScaler
   │
   └── ElasticNet
   ↓
RandomizedSearchCV
   ↓
Random parameter combinations
   ↓
K-Fold Cross-Validation
   ↓
Compare scores
   ↓
Best Parameters
   ↓
Best Estimator
   ↓
Final Test
```

---

# Parameter Naming

Nothing changes from GridSearchCV.

If our Pipeline is:

```python
Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet())
])
```

we still use:

```text
model__alpha
model__l1_ratio
```

Remember:

```text
Pipeline step
     +
     __
     +
parameter

model + __ + alpha

        ↓

model__alpha
```

---

# Basic Pattern

```python
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=10,
    cv=kf,
    scoring="r2",
    random_state=42
)
```

Then:

```python
random_search.fit(X_train, y_train)
```

Results:

```python
random_search.best_params_
random_search.best_score_
random_search.best_estimator_
```

---

# GridSearchCV

```text
PARAMETER SPACE
      ↓
TRY EVERYTHING
      ↓
CV EACH COMBINATION
      ↓
BEST
```

# RandomizedSearchCV

```text
PARAMETER SPACE
      ↓
RANDOMLY SAMPLE
      ↓
TRY n_iter COMBINATIONS
      ↓
CV EACH COMBINATION
      ↓
BEST FOUND
```

---

# When Should I Use Which?

### Small parameter space

```text
Few parameters
+
Few possible values

        ↓

GridSearchCV
```

Example:

```text
4 alpha values
×
3 ratios

= 12 combinations
```

Testing all 12 is easy.

---

### Large parameter space

```text
Many parameters
+
Many possible values

        ↓

RandomizedSearchCV
```

Example:

```text
10 × 10 × 10 × 10

= 10,000 combinations
```

Instead of testing everything:

```text
RandomizedSearchCV
n_iter = 50
```

tests only 50 sampled combinations.

---

# Important Difference in Arguments

GridSearchCV:

```python
GridSearchCV(
    param_grid=params
)
```

RandomizedSearchCV:

```python
RandomizedSearchCV(
    param_distributions=params
)
```

Easy memory:

```text
GRID
↓
param_grid


RANDOM
↓
param_distributions
```

---

# 🧠 Quick Recall

```text
             HYPERPARAMETER TUNING
                       │
            ┌──────────┴──────────┐
            ▼                     ▼
       GridSearchCV       RandomizedSearchCV
            │                     │
        ALL combos            RANDOM combos
            │                     │
       Exhaustive               n_iter
            │                     │
        Small grid            Large space
            │                     │
            └──────────┬──────────┘
                       ▼
                 Cross-Validation
                       ↓
                 Best Parameters
                       ↓
                  Best Model
```

---

# 10-Second Memory

```text
GRID
→ Try ALL


RANDOM
→ Try SOME


n_iter
→ HOW MANY combinations to try
```

## Golden Rule

> **Use GridSearchCV when the search space is small. Use RandomizedSearchCV when testing every combination would be expensive.**


In [59]:
model = Pipeline([
    ("scalar", StandardScaler()),
    ("model", ElasticNet(max_iter=10000))
])

In [60]:
params = {
    "model__alpha": [
        0.0001, 0.001, 0.005, 0.01,
        0.05, 0.1, 0.5, 1, 5, 10
    ],

    "model__l1_ratio": [
        0.1, 0.2, 0.3, 0.4, 0.5,
        0.6, 0.7, 0.8, 0.9
    ]
}

In [61]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=params,
    n_iter=20,
    cv=kf,
    scoring="r2",
    random_state=42
)

In [62]:
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('scalar', StandardScaler()),
                                             ('model',
                                              ElasticNet(max_iter=10000))]),
                   n_iter=20,
                   param_distributions={'model__alpha': [0.0001, 0.001, 0.005,
                                                         0.01, 0.05, 0.1, 0.5,
                                                         1, 5, 10],
                                        'model__l1_ratio': [0.1, 0.2, 0.3, 0.4,
                                                            0.5, 0.6, 0.7, 0.8,
                                                            0.9]},
                   random_state=42, scoring='r2')

In [63]:
print("Best Parameters:", random_search.best_params_)
print("Best CV R²:", random_search.best_score_)

Best Parameters: {'model__l1_ratio': 0.1, 'model__alpha': 0.005}
Best CV R²: 0.29205294325366454


In [64]:
import numpy as np

params = {
    "model__alpha": np.logspace(-4, 1, 100),
    "model__l1_ratio": np.linspace(0.1, 0.9, 50)
}

In [65]:
RandomizedSearchCV(
    model,
    param_distributions=params,
    n_iter=50,
    cv=kf,
    scoring="r2",
    random_state=42
)

RandomizedSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('scalar', StandardScaler()),
                                             ('model',
                                              ElasticNet(max_iter=10000))]),
                   n_iter=50,
                   param_distributions={'model__alpha': array([1.00000000e-04, 1.12332403e-04, 1.26185688e-04, 1.41747416e-04,
       1.59228279e-04, 1.78864953e-04, 2.00923300e-04, 2.25701972e-04,
       2.53536449e-04,...
       0.42653061, 0.44285714, 0.45918367, 0.4755102 , 0.49183673,
       0.50816327, 0.5244898 , 0.54081633, 0.55714286, 0.57346939,
       0.58979592, 0.60612245, 0.62244898, 0.63877551, 0.65510204,
       0.67142857, 0.6877551 , 0.70408163, 0.72040816, 0.73673469,
       0.75306122, 0.76938776, 0.78571429, 0.80204082, 0.81836735,
       0.83469388, 0.85102041, 0.86734694, 0.88367347, 0.9       ])},
                   random_state=42, scoring='r2')

In [67]:
random_search.fit(X_train, y_train)
print("Best Parameters:", random_search.best_params_)

print("Best CV R²:", random_search.best_score_)

Best Parameters: {'model__l1_ratio': 0.1, 'model__alpha': 0.005}
Best CV R²: 0.29205294325366454


# Scikit-Learn Pipeline — Practical Guide

## What is a Pipeline?

A `Pipeline` connects multiple Machine Learning steps into **one object**.

Instead of manually doing:

```text
Missing Values
      ↓
Scaling
      ↓
Model Training
      ↓
Prediction
```

we can create:

```text
Pipeline
│
├── Imputer
├── Scaler
└── Model
```

Then:

```python
pipeline.fit(X_train, y_train)
```

can execute the required training steps automatically.

---

# Why Do We Need Pipeline?

Without Pipeline:

```python
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
```

This works, but as preprocessing becomes larger it becomes easier to make mistakes.

With Pipeline:

```python
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
```

The Pipeline handles the intermediate transformations.

---

# The Most Important Idea

## During Training

```text
X_train
   ↓
Imputer
fit + transform
   ↓
Scaler
fit + transform
   ↓
Model
fit
```

So:

```python
pipeline.fit(X_train, y_train)
```

conceptually performs:

```text
Imputer.fit_transform(X_train)

        ↓

Scaler.fit_transform(...)

        ↓

Model.fit(...)
```

---

# During Prediction

Now suppose completely unseen data arrives.

```python
pipeline.predict(X_new)
```

The flow becomes:

```text
X_new
   ↓
Imputer
transform ONLY
   ↓
Scaler
transform ONLY
   ↓
Model
predict
   ↓
Prediction
```

The preprocessing steps do NOT relearn their statistics from the new data.

They reuse what they learned during training.

---

# Example — StandardScaler

During training:

```text
X_train
   ↓
StandardScaler.fit()
   ↓
Learns:

mean
standard deviation
```

Then:

```text
X_train
   ↓
StandardScaler.transform()
   ↓
Scaled training data
```

During prediction:

```text
X_test
   ↓
StandardScaler.transform()
   ↓
Uses TRAINING mean/std
```

NOT:

```text
X_test
   ↓
StandardScaler.fit() ❌
```

The test data should not teach the scaler anything.

---

# Example — SimpleImputer

Suppose:

```text
Age

20
25
NaN
35
40
```

During training:

```text
SimpleImputer.fit()
        ↓
Learns median
        ↓
30
```

Then it replaces missing values using that learned value.

For future/test data:

```text
22
NaN
45
```

the imputer uses the value learned from the **training data**.

It does NOT calculate a new median from the test data.

---

# Why Pipeline Helps Prevent Leakage

Wrong approach:

```text
FULL DATASET
     ↓
Scaler.fit()
     ↓
Train/Test Split
```

The scaler has already seen information from what later becomes the test set.

That is data leakage.

Better:

```text
FULL DATASET
     ↓
Train/Test Split
    /          \
 TRAIN         TEST
   ↓
Pipeline
   ↓
Scaler learns
from TRAIN
only
```

---

# Basic Pipeline Structure

```python
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("step1", transformer),
    ("step2", transformer),
    ("model", estimator)
])
```

Example:

```python
pipeline = Pipeline([
    ("imputer", SimpleImputer()),
    ("scaler", StandardScaler()),
    ("model", Ridge())
])
```

---

# Pipeline Rule

All steps before the final step must be **transformers**.

They need to support transformation of the data.

Examples:

```text
SimpleImputer
StandardScaler
PolynomialFeatures
OneHotEncoder
PCA
```

The final step is commonly the estimator/model:

```text
LinearRegression
Ridge
Lasso
ElasticNet
LogisticRegression
RandomForest
etc.
```

---

# Step Names

In:

```python
pipeline = Pipeline([
    ("imputer", SimpleImputer()),
    ("scaler", StandardScaler()),
    ("model", Ridge())
])
```

these:

```text
imputer
scaler
model
```

are names WE choose.

They allow us to access parts of the Pipeline.

For example:

```python
pipeline.named_steps["model"]
```

returns the model.

And:

```python
pipeline.named_steps["scaler"]
```

returns the scaler.

---

# Pipeline Parameters

Suppose:

```text
Pipeline step = model
Model parameter = alpha
```

Then:

```text
model__alpha
```

means:

```text
model
  ↓
 __
  ↓
alpha
```

This is why GridSearchCV used:

```python
{
    "model__alpha": [0.01, 0.1, 1, 10]
}
```

---

# Pipeline + Cross-Validation

Pipeline becomes especially important with Cross-Validation.

```text
Fold 1
│
├── Training portion
│      ↓
│   Fit scaler
│      ↓
│   Fit model
│
└── Validation portion
       ↓
    Transform using
    training scaler
       ↓
    Predict
```

Then everything is fitted again independently for Fold 2, Fold 3, etc.

This helps prevent preprocessing leakage between folds.

---

# Pipeline Mental Model

Think of Pipeline as a factory conveyor belt:

```text
RAW DATA
   │
   ▼
┌─────────────┐
│   IMPUTER   │
│ Fix missing │
└──────┬──────┘
       ▼
┌─────────────┐
│   SCALER    │
│ Scale data  │
└──────┬──────┘
       ▼
┌─────────────┐
│    MODEL    │
│ Learn       │
└──────┬──────┘
       ▼
  PREDICTION
```

The same preprocessing path is reused when new data arrives.

---

# 🧠 Quick Recall

```text
TRAINING

X_train
   ↓
Transformer.fit_transform()
   ↓
Transformer.fit_transform()
   ↓
Model.fit()


PREDICTION

X_test
   ↓
Transformer.transform()
   ↓
Transformer.transform()
   ↓
Model.predict()
```

---

# Why Pipeline?

```text
Pipeline
   │
   ├── Cleaner code
   ├── Consistent preprocessing
   ├── Helps prevent leakage
   ├── Works with Cross-Validation
   ├── Works with GridSearchCV
   └── Easier deployment
```

## Golden Rule

> Fit preprocessing only using training data. Apply the learned transformations to validation, test, and future data.
